# 11장. 지식 증류 (Knowledge Distillation)

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


Teacher(Base)는 라즈베리파이에서 돌리기엔 무겁습니다. 그래서 작은 **Student(Small)**를 만듭니다. 이때 Student가 정답 라벨만 보고 배우는 대신, **Teacher의 판단까지** 보고 배우게 하는 것이 지식 증류입니다.

## 11.1 정답(hard) vs Teacher의 확신(soft)

같은 "고장" 윈도우라도 Teacher가 보기에 더 애매한 것이 있습니다.

| 윈도우 | 정답 | Teacher 확률 [정상, 고장] |
|---|---|---|
| A | 고장 | [0.02, 0.98] ← 확실한 고장 |
| B | 고장 | [0.35, 0.65] ← 정상과 좀 닮은 고장 |

정답만 보면 A와 B는 똑같이 "고장"입니다. Teacher의 확률을 보면 **B가 정상과 비슷한 면이 있다**는 정보까지 전달됩니다. 이 추가 정보를 "어두운 지식(dark knowledge)"이라고도 부릅니다.

## 11.2 온도(temperature): 확률을 부드럽게

Teacher가 너무 확신하면 확률이 [0.001, 0.999]처럼 한쪽으로 쏠려 정보가 거의 없습니다. logits를 **온도 T로 나눈 뒤** softmax 하면 분포가 부드러워집니다.

In [2]:
import torch, torch.nn.functional as F
teacher_logits = torch.tensor([[-3.0, 3.5]])
for t in (1, 2, 4):
    print(f"T={t}: {F.softmax(teacher_logits / t, dim=1).numpy().round(3)}")

T=1: [[0.002 0.998]]
T=2: [[0.037 0.963]]
T=4: [[0.165 0.835]]


## 11.3 KD loss

```text
KD loss = 0.5 × CE(Student, 정답)                       ← 정답 맞히기 (hard)
        + 0.5 × KL(Teacher_T ‖ Student_T) × T²          ← Teacher 분포 따라하기 (soft)
```

- **KL divergence**: 두 확률 분포가 얼마나 다른지. Teacher와 같아지면 0
- **× T²**: 온도로 나누면 기울기가 1/T² 로 작아지므로 다시 키워 균형을 맞춥니다
- 이 프로젝트: T = 2, 정답 비중 0.5

In [3]:
import os, warnings
os.environ["HF_HUB_OFFLINE"] = "1"                 # 인터넷 없이 로컬 모델 폴더만 사용
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"    # Mac: MPS가 모르는 연산은 CPU로
warnings.filterwarnings("ignore")
import torch
from ch09_train_teacher import pick_device
DEVICE = pick_device()
print("장치:", DEVICE)

장치: mps


In [4]:
import ch11_distill as ch11
labels = torch.tensor([1])
t_logits = torch.tensor([[-0.3, 0.3]])                 # Teacher: 고장 쪽이지만 애매함
for name, s in [("Teacher와 비슷", [[-0.3, 0.3]]), ("지나치게 확신", [[-4.0, 4.0]]), ("틀림", [[2.0, -2.0]])]:
    s = torch.tensor(s)
    print(f"Student {name:9s}: CE {F.cross_entropy(s, labels).item():.3f}  KD loss {ch11.kd_loss(s, t_logits, labels).item():.3f}")

Student Teacher와 비슷: CE 0.437  KD loss 0.219
Student 지나치게 확신  : CE 0.000  KD loss 2.077
Student 틀림       : CE 4.018  KD loss 3.197


Teacher가 애매하게 본 샘플에 Student가 지나치게 확신하면, 정답은 맞아도 KD loss가 커집니다. Student가 Teacher처럼 조심스러워지도록 유도하는 것입니다.

## 11.4 Teacher 예측 캐시

Student를 학습할 때마다 무거운 Teacher를 다시 돌리면 낭비입니다. 그래서 **Teacher의 logits를 한 번만 계산해 `.npz`로 저장**합니다.

이때 6장의 **데이터 지문**과 샘플 ID를 함께 저장합니다. 캐시를 만든 뒤 데이터가 바뀌거나 순서가 달라지면, Teacher 예측과 윈도우의 짝이 어긋난 채로 학습이 **조용히** 진행되기 때문입니다. 불러올 때 지문이 다르면 에러를 냅니다.

## 11.5 Student는 head만, 윈도우 단위로

- Student는 7장처럼 **몸통 고정, head(1,026개)만** 학습합니다.
- Teacher 캐시가 윈도우 단위이므로, Student도 **청크 8개 평균 logits**(윈도우 단위)로 loss를 계산합니다.

## 11.6 실행 (약 1분): 캐시 → Student 20 epoch

In [5]:
%run code/ch11_distill.py

[1] Teacher 예측 캐시


Loading weights from local directory


  teacher_train.npz: 윈도우 84개, Teacher 정확도 1.000


  teacher_validation.npz: 윈도우 28개, Teacher 정확도 1.000
[2] Student(head) 증류 학습


Loading weights from local directory


...

  epoch 15  KD loss 0.0311  validation 28/28 loss 0.0082 [[14, 0], [0, 14]]  ← best 저장


  epoch 16  KD loss 0.0264  validation 28/28 loss 0.0078 [[14, 0], [0, 14]]  ← best 저장


  epoch 17  KD loss 0.0277  validation 28/28 loss 0.0072 [[14, 0], [0, 14]]  ← best 저장


  epoch 19  KD loss 0.0292  validation 28/28 loss 0.0069 [[14, 0], [0, 14]]  ← best 저장


  epoch 20  KD loss 0.0264  validation 28/28 loss 0.0067 [[14, 0], [0, 14]]  ← best 저장


In [6]:
import numpy as np
cache = np.load(ROOT / "outputs/manual/student/cache/teacher_train.npz")
print("캐시 내용:", cache.files)
print("첫 샘플:", str(cache["sample_ids"][0]), "라벨", int(cache["labels"][0]), "Teacher logits", cache["logits"][0].round(3))
print("데이터 지문:", str(cache["dataset_fingerprint"])[:16], "…")

캐시 내용: ['logits', 'labels', 'sample_ids', 'dataset_fingerprint', 'teacher_sha256']
첫 샘플: K001/N15_M07_F04_K001_1_windows.npy:0 라벨 0 Teacher logits [ 2.897 -2.911]
데이터 지문: 20de101d5879f719 …


In [7]:
from ch06_dataset import WindowDataset
import json
split = json.loads((ROOT / "outputs/manual/split_holdout.json").read_text())
files = [ROOT / f for f in split["train"]]
try:
    ch11.DistillDataset(list(reversed(files)), ROOT / "outputs/manual/student/cache/teacher_train.npz")
except ValueError as e:
    print("순서가 바뀐 데이터는 거부:", e)

순서가 바뀐 데이터는 거부: 캐시를 만든 뒤 데이터가 바뀌었습니다. cache_teacher를 다시 실행하세요.


## 11.7 전체 증류 (터미널에서)

```bash
python docs/manual/code/ch11_distill.py --split data/splits/bearing_holdout_v3.json \
    --teacher outputs/manual/teacher_full/teacher_lora_best.pt --epochs 20 --learning-rate 1e-4 \
    --output-dir outputs/manual/student_full
```
실제 프로젝트 명령: `bearing-kd cache-teacher ...` → `bearing-kd train-student ...` (Mac MPS 기준 약 25분)

> **증류의 한계:** Student는 Teacher가 아는 것만 배울 수 있습니다. Teacher가 못 맞히는 고장은 Student도 못 맞힙니다(12장). 또 Student는 head만 학습하므로, 정답 라벨만으로 학습한 head와 성능 차이가 크지 않을 수 있습니다. 증류의 실제 효과는 둘을 비교해 봐야 알 수 있고, 이 프로젝트에서는 아직 비교하지 않았습니다.

## 11장 완성 코드: `code/ch11_distill.py`

In [8]:
from IPython.display import Code
Code(filename=str(CODE / "ch11_distill.py"), language="python")

"""11장 완성 코드: 지식 증류(KD). Teacher의 예측을 캐시로 저장하고, Student(MOMENT Small)의 head만 학습한다.

순서:
    1) cache_teacher : Teacher로 train/validation 윈도우를 예측해 logits를 .npz로 저장 (데이터 지문 포함)
    2) train_student : Small의 head만, "정답 + Teacher의 확신 정도"를 함께 보며 학습

실행 (프로젝트 루트에서, 9장 결과 사용):
    python docs/manual/code/ch11_distill.py
    # 전체 (실제 프로젝트 split, 프로젝트와 같은 설정)
    python docs/manual/code/ch11_distill.py --split data/splits/bearing_holdout_v3.json \
        --teacher outputs/manual/teacher_full/teacher_lora_best.pt --epochs 20 --learning-rate 1e-4 \
        --output-dir outputs/manual/student_full
"""

import argparse
import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from ch06_dataset import WindowDataset
from ch07_moment import ROOT, load_moment
from ch09_train_teacher import evaluate, pick_device, seed_everything, window_logits
from ch10_checkpoint import load_full

TEMPERATURE = 2.0   # 확률 분포를 얼마나 "부드럽게" 펼칠지
HARD_WEIGHT = 0.5   # 정답 손실 비중 (나머지 0.5는 Teacher 따라하기)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


@torch.no_grad()
def cache_teacher(teacher_ckpt, split, model_dir, out_dir, device):
    """Teacher 예측을 한 번만 계산해 저장한다. Student를 여러 번 학습해도 Teacher를 다시 돌릴 필요가 없다."""
    teacher = load_full(model_dir, teacher_ckpt).to(device)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name in ("train", "validation"):
        ds = WindowDataset([ROOT / f for f in split[name]])
        logits = torch.cat([window_logits(teacher, x.to(device)).cpu()
                            for x, _ in DataLoader(ds, batch_size=32)]).numpy()
        np.savez_compressed(out_dir / f"teacher_{name}.npz", logits=logits, labels=ds.labels,
                            sample_ids=np.array(ds.sample_ids), dataset_fingerprint=ds.fingerprint,
                            teacher_sha256=sha256(teacher_ckpt))
        acc = (logits.argmax(1) == ds.labels).mean()
        print(f"  teacher_{name}.npz: 윈도우 {len(ds)}개, Teacher 정확도 {acc:.3f}")


class DistillDataset(WindowDataset):
    """윈도우 + 정답 + 같은 윈도우에 대한 Teacher logits. 캐시와 데이터가 어긋나면 에러."""

    def __init__(self, files, cache_path):
        super().__init__(files)
        cache = np.load(cache_path)
        if str(cache["dataset_fingerprint"]) != self.fingerprint:
            raise ValueError("캐시를 만든 뒤 데이터가 바뀌었습니다. cache_teacher를 다시 실행하세요.")
        assert list(cache["sample_ids"]) == self.sample_ids and np.array_equal(cache["labels"], self.labels)
        self.teacher_logits = torch.from_numpy(cache["logits"]).float()

    def __getitem__(self, i):
        x, y = super().__getitem__(i)
        return x, y, self.teacher_logits[i]


def kd_loss(student_logits, teacher_logits, labels, t=TEMPERATURE, hard_weight=HARD_WEIGHT):
    """0.5 × (정답 맞히기) + 0.5 × (Teacher의 확률 분포 따라하기)."""
    hard = F.cross_entropy(student_logits, labels)
    soft = F.kl_div(F.log_softmax(student_logits / t, dim=1),   # Student 분포 (온도로 부드럽게)
                    F.softmax(teacher_logits / t, dim=1),        # Teacher 분포 (온도로 부드럽게)
                    reduction="batchmean") * t ** 2              # t²: 온도 때문에 작아진 기울기 크기 보정
    return hard_weight * hard + (1 - hard_weight) * soft


def train_student(split, cache_dir, model_dir, out_dir, epochs, lr, batch_size, seed, device):
    seed_everything(seed)
    train_ds = DistillDataset([ROOT / f for f in split["train"]], cache_dir / "teacher_train.npz")
    valid_ds = WindowDataset([ROOT / f for f in split["validation"]])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        student = load_moment(model_dir, seed=seed).to(device)   # 몸통 고정, head만 학습 (7장)
    params = [p for p in student.parameters() if p.requires_grad]
    print(f"  Student 학습 대상 {sum(p.numel() for p in params):,}개 (head만)")
    optimizer = torch.optim.AdamW(params, lr=lr)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator